In [0]:
from pyspark.sql import functions as F

import os
current_dir = os.path.dirname(os.path.abspath("__file__"))
path = f"{current_dir}/eurostat_mean-median_2026.csv"
df_raw = spark.read.csv(path, header=True, inferSchema=True)

df_filtered = df_raw.filter(
    (F.col("TIME_PERIOD") == 2022) & 
    (F.col("nace_r2") == "B-S_X_O")
)

# Map country codes to Polish names
country_names_pl = {
    "AT": "Austria", "BE": "Belgia", "BG": "Bułgaria", "HR": "Chorwacja",
    "CY": "Cypr", "CZ": "Czechy", "DK": "Dania", "EE": "Estonia",
    "FI": "Finlandia", "FR": "Francja", "DE": "Niemcy", "EL": "Grecja",
    "HU": "Węgry", "IE": "Irlandia", "IT": "Włochy", "LV": "Łotwa",
    "LT": "Litwa", "LU": "Luksemburg", "MT": "Malta", "NL": "Holandia",
    "PL": "Polska", "PT": "Portugalia", "RO": "Rumunia", "SK": "Słowacja",
    "SI": "Słowenia", "ES": "Hiszpania", "SE": "Szwecja", "CH": "Szwajcaria",
    "AL": "Albania", "BA": "Bośnia i Hercegowina", "IS": "Islandia",
    "MK": "Macedonia Północna", "NO": "Norwegia", "RS": "Serbia"
}

# Map country codes to English names
country_names_en = {
    "AT": "Austria", "BE": "Belgium", "BG": "Bulgaria", "HR": "Croatia",
    "CY": "Cyprus", "CZ": "Czechia", "DK": "Denmark", "EE": "Estonia",
    "FI": "Finland", "FR": "France", "DE": "Germany", "EL": "Greece",
    "HU": "Hungary", "IE": "Ireland", "IT": "Italy", "LV": "Latvia",
    "LT": "Lithuania", "LU": "Luxembourg", "MT": "Malta", "NL": "Netherlands",
    "PL": "Poland", "PT": "Portugal", "RO": "Romania", "SK": "Slovakia",
    "SI": "Slovenia", "ES": "Spain", "SE": "Sweden", "CH": "Switzerland",
    "AL": "Albania", "BA": "Bosnia and Herzegovina", "IS": "Iceland",
    "MK": "North Macedonia", "NO": "Norway", "RS": "Serbia"
}

# Convert dictionaries to Spark mappings
country_mapping_pl = F.create_map([F.lit(x) for pair in country_names_pl.items() for x in pair])
country_mapping_en = F.create_map([F.lit(x) for pair in country_names_en.items() for x in pair])

# Pivot data
df_pivoted = df_filtered.groupBy(
    F.col("geo").alias("Kod_Kraju")
).pivot("indic_se", ["MEAN_E_EUR", "MED_E_EUR"]).agg(F.first("OBS_VALUE"))


df_final = df_pivoted.withColumnRenamed("MEAN_E_EUR", "Srednia") \
                     .withColumnRenamed("MED_E_EUR", "Mediana") \
                     .withColumn("Kraj", country_mapping_pl[F.col("Kod_Kraju")]) \
                     .withColumn("Country_EN", country_mapping_en[F.col("Kod_Kraju")]) \
                     .withColumn("Rozwarstwienie_Pct", F.round(((F.col("Srednia") - F.col("Mediana")) / F.col("Srednia")) * 100, 2)) \
                     .filter(F.col("Srednia").isNotNull() & F.col("Mediana").isNotNull()) \
                     .filter(~F.col("Kod_Kraju").isin(["EA19", "EA20", "EU27_2020"])) \
                     .orderBy(F.col("Rozwarstwienie_Pct").desc())

display(df_final)

Databricks visualization. Run in Databricks to view.

In [0]:
import matplotlib.pyplot as plt
import numpy as np

# Convert to pandas for plotting, filter out nulls
df_plot = df_final.filter(F.col("Country_EN").isNotNull()).select("Country_EN", "Rozwarstwienie_Pct").toPandas()

# Sort by inequality percentage
df_plot = df_plot.sort_values("Rozwarstwienie_Pct", ascending=True)

# Calculate average
avg_inequality = df_plot["Rozwarstwienie_Pct"].mean()

# Create colors: red for Poland, steelblue for others
colors = ['red' if country == 'Poland' else 'steelblue' for country in df_plot['Country_EN']]

# Create horizontal bar chart
fig, ax = plt.subplots(figsize=(10, 12))
ax.barh(df_plot['Country_EN'], df_plot['Rozwarstwienie_Pct'], color=colors)

# Add dashed line for average
ax.axvline(avg_inequality, color='black', linestyle='--', linewidth=2, label=f'Average: {avg_inequality:.2f}%')

# Labels and title
ax.set_xlabel('Income Inequality (%)', fontsize=12)
ax.set_ylabel('Country', fontsize=12)
ax.set_title('Income Inequality by Country (2022)\nMean vs Median Earnings Gap', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

In [0]:
import matplotlib.pyplot as plt
import numpy as np

# Get data to Pandas, filter out nulls for Polish country names
df_plot = df_final.filter(F.col("Kraj").isNotNull()).select("Kraj", "Rozwarstwienie_Pct").toPandas()

# Sort by inequality percentage (ascending, so highest is on top)
df_plot = df_plot.sort_values("Rozwarstwienie_Pct", ascending=True)

# Calculate European average
avg_inequality = df_plot["Rozwarstwienie_Pct"].mean()

# Set dark background style
plt.style.use('dark_background')

# Create chart with deep, elegant background (charcoal)
fig, ax = plt.subplots(figsize=(10, 12), facecolor='#121212')
ax.set_facecolor('#121212')

# Premium color palette: elegant turquoise for Europe, coral red for Poland
colors = ['#ff5252' if kraj == 'Polska' else '#26a69a' for kraj in df_plot['Kraj']]

# Draw horizontal bars
bars = ax.barh(df_plot['Kraj'], df_plot['Rozwarstwienie_Pct'], color=colors, height=0.75, edgecolor='none')

# Add vertical average line in muted gold color
ax.axvline(avg_inequality, color='#ffb300', linestyle='--', linewidth=1.5, 
           label=f'European average: {avg_inequality:.2f}%')

# Title and axis labels (in Polish)
ax.set_xlabel('Income inequality (%)', fontsize=12, fontweight='bold', color='#e0e0e0', labelpad=10)
ax.set_ylabel('Country', fontsize=12, fontweight='bold', color='#e0e0e0', labelpad=10)
ax.set_title('Income Inequality in Europe (2022)\nPercentage gap between mean and median earnings', 
             fontsize=14, fontweight='bold', color='#ffffff', pad=25)

# Remove unnecessary frames (spines) for modern, frameless look
for spine in ['top', 'right', 'left', 'bottom']:
    ax.spines[spine].set_visible(False)

# Configure grid and axes (delicate helper lines)
ax.xaxis.grid(True, which='major', color='#2c2c2c', linestyle=':', alpha=0.6)
ax.tick_params(colors='#b0b0b0', labelsize=11)

# Add numeric values at the end of each bar (premium effect)
for bar in bars:
    width = bar.get_width()
    ax.text(width + 0.3, bar.get_y() + bar.get_height()/2, f'{width:.2f}%', 
            va='center', ha='left', fontsize=10, color='#b0b0b0', fontweight='bold')

# Configure elegant legend
ax.legend(facecolor='#1e1e1e', edgecolor='#2c2c2c', loc='lower right', fontsize=11)

plt.tight_layout()

# Save to file (in Databricks you can use plt.show() at the end)
plt.savefig('wykres_premium_dark.png', facecolor=fig.get_facecolor(), bbox_inches='tight', dpi=300)

In [0]:
from pyspark.sql import functions as F

# 1. Official happiness indicators from World Happiness Report 2022
# Data perfectly synchronized with the year of your salary data
happiness_data_2022 = {
    "FI": 7.821, "DK": 7.636, "IS": 7.557, "CH": 7.240, "NL": 7.403,
    "LU": 7.404, "SE": 7.384, "NO": 7.365, "AT": 7.163, "DE": 6.892,
    "FR": 6.687, "IE": 7.041, "BE": 6.805, "CZ": 6.920, "PL": 6.125,
    "ES": 6.476, "IT": 6.405, "SI": 6.630, "SK": 6.044, "LT": 6.446,
    "LV": 6.180, "EE": 6.341, "RO": 6.471, "HU": 5.371, "HR": 6.125,
    "CY": 6.221, "BG": 5.371, "EL": 5.948, "PT": 6.015, "MT": 6.447,
    "RS": 6.178, "BA": 5.377, "MK": 5.159, "AL": 5.200
}

# Convert dictionary to Spark mapping
happiness_mapping_2022 = F.create_map([F.lit(x) for pair in happiness_data_2022.items() for x in pair])

# 2. Add column with 2022 indicator to your DataFrame (df_final)
df_with_happiness_2022 = df_final.withColumn("Happiness_Score", happiness_mapping_2022[F.col("Kod_Kraju")]) \
                                  .filter(F.col("Happiness_Score").isNotNull())

# 3. Calculate Pearson correlation coefficient for synchronized data
r_2022 = df_with_happiness_2022.stat.corr("Rozwarstwienie_Pct", "Happiness_Score")

print(f"--- CORRELATION ANALYSIS FOR YEAR 2022 ---")
print(f"Pearson correlation coefficient (r): {round(r_2022, 4)}")
print(f"Number of countries from your CSV: {df_with_happiness_2022.count()}")
print("-------------------------------------------")

# Display result table for new scatter plot
# display(df_with_happiness_2022.select("Kraj", "Rozwarstwienie_Pct", "Happiness_Score", "Srednia", "Mediana"))
display(df_with_happiness_2022.select("Kod_Kraju", "Rozwarstwienie_Pct", "Happiness_Score"))

Databricks visualization. Run in Databricks to view.